In [37]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [38]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
valid_df = pd.read_csv("valid.csv")

In [39]:
train_df.columns

Index(['ID', 'Label', 'Statement', 'Subject(s)', 'Speaker',
       'Speaker's job title', 'State', 'Party', 'Barely true counts',
       'False counts', 'Half true counts', 'Mostly true counts',
       'Pants on fire counts', 'Context'],
      dtype='object')

Preprocess text

In [40]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ravindu_pathirana/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [41]:
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(tokens)

train_df["Statement"] = train_df["Statement"].apply(preprocess_text)
test_df["Statement"] = test_df["Statement"].apply(preprocess_text)
valid_df["Statement"] = valid_df["Statement"].apply(preprocess_text)

Preprocess label

In [42]:
def convert_label(label):
    if label in ['pants-fire', 'false', 'barely-true']:
        return 0  # fake
    else:
        return 1  # real

train_df['Label'] = train_df['Label'].apply(convert_label)
test_df['Label'] = test_df['Label'].apply(convert_label)
valid_df['Label'] = valid_df['Label'].apply(convert_label)

### Text Only Baseline 

In [43]:
X_train = train_df["Statement"]
y_train = train_df["Label"]

X_test = test_df["Statement"]
y_test = test_df["Label"]

X_valid = valid_df["Statement"]
y_valid = valid_df["Label"]

TF-IDF feature extraction

In [44]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
X_valid_tfidf = vectorizer.transform(X_valid)

Majority Baseline (Wang 2017)

In [45]:
majority = DummyClassifier(strategy="most_frequent")
majority.fit(X_train_tfidf, y_train)

y_pred = majority.predict(X_test_tfidf)
print("Majority Accuracy:", accuracy_score(y_test, y_pred))

Majority Accuracy: 0.7600631412786109


Naive Bayes Model

In [46]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

y_pred = nb.predict(X_test_tfidf)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred))

Naive Bayes Accuracy: 0.7545382794001578


Logistic Regression Model

In [47]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)

y_pred = lr.predict(X_test_tfidf)

print("LR Accuracy:", accuracy_score(y_test, y_pred))

LR Accuracy: 0.7576953433307024


Support Vector Machine Model

In [48]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)

y_pred = svm.predict(X_test_tfidf)

print("SVM Accuracy:", accuracy_score(y_test, y_pred))

SVM Accuracy: 0.7103393843725335


Random Forest

In [49]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train_tfidf, y_train)

y_pred = rf.predict(X_test_tfidf)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))

Random Forest Accuracy: 0.7379636937647988


Evaluation Function

In [50]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(y_true, y_pred, name):
    print(f"\n{name}")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred))

evaluate(y_test, lr.predict(X_test_tfidf), "Logistic Regression")
evaluate(y_test, svm.predict(X_test_tfidf), "SVM")
evaluate(y_test, nb.predict(X_test_tfidf), "Naive Bayes")
evaluate(y_test, rf.predict(X_test_tfidf), "Random Forest")


Logistic Regression
[[ 12 292]
 [ 15 948]]
              precision    recall  f1-score   support

           0       0.44      0.04      0.07       304
           1       0.76      0.98      0.86       963

    accuracy                           0.76      1267
   macro avg       0.60      0.51      0.47      1267
weighted avg       0.69      0.76      0.67      1267


SVM
[[ 59 245]
 [122 841]]
              precision    recall  f1-score   support

           0       0.33      0.19      0.24       304
           1       0.77      0.87      0.82       963

    accuracy                           0.71      1267
   macro avg       0.55      0.53      0.53      1267
weighted avg       0.67      0.71      0.68      1267


Naive Bayes
[[  5 299]
 [ 12 951]]
              precision    recall  f1-score   support

           0       0.29      0.02      0.03       304
           1       0.76      0.99      0.86       963

    accuracy                           0.75      1267
   macro avg       0

In [51]:
models = {
    "LR": lr,
    "SVM": svm,
    "NB": nb,
    "RF": rf
}

for name, model in models.items():
    y_pred = model.predict(X_valid_tfidf)
    acc = accuracy_score(y_valid, y_pred)
    print(f"{name} Validation Accuracy: {acc}")

LR Validation Accuracy: 0.7289719626168224
SVM Validation Accuracy: 0.6814641744548287
NB Validation Accuracy: 0.7258566978193146
RF Validation Accuracy: 0.7110591900311527


In [52]:
for name, model in models.items():
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name} Test Accuracy: {acc}")

LR Test Accuracy: 0.7576953433307024
SVM Test Accuracy: 0.7103393843725335
NB Test Accuracy: 0.7545382794001578
RF Test Accuracy: 0.7379636937647988
